# 💳 Task 2: Credit Card Fraud Detection
**CodSoft ML Internship** — Random Forest with Imbalanced Data Handling

> Replace the dummy dataset with `pd.read_csv('creditcard.csv')` (Kaggle dataset)

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score
)
plt.rcParams.update({'figure.dpi':110})
print('✅ Imports done')

## 📂 Load Dataset

In [ ]:
# ── Replace with your real CSV path ──────────────────────
# df = pd.read_csv(r'D:\codsoft\creditcard.csv')

# Dummy dataset (mirrors Kaggle creditcard.csv structure)
def make_fraud_data(n=15000, fraud_rate=0.02, seed=42):
    rng = np.random.default_rng(seed)
    nf = int(n*fraud_rate); nn = n-nf
    nor = pd.DataFrame(rng.standard_normal((nn,28)), columns=[f'V{i}' for i in range(1,29)])
    fr  = pd.DataFrame(rng.standard_normal((nf,28))+3, columns=[f'V{i}' for i in range(1,29)])
    nor['Amount'] = rng.exponential(50, nn); fr['Amount'] = rng.exponential(200, nf)
    nor['Time'] = rng.uniform(0,172800,nn); fr['Time'] = rng.uniform(0,172800,nf)
    nor['Class'] = 0; fr['Class'] = 1
    return pd.concat([nor, fr]).sample(frac=1, random_state=seed).reset_index(drop=True)

df = make_fraud_data()
print(f'Shape: {df.shape} | Fraud rate: {df["Class"].mean()*100:.2f}%')
df.head()

## 🔍 EDA

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Class distribution
vc = df['Class'].value_counts()
axes[0].bar(['Normal','Fraud'], vc.values, color=['#3498db','#e74c3c'], edgecolor='black')
for i,v in enumerate(vc.values): axes[0].text(i, v+30, f'{v:,}', ha='center', fontweight='bold')
axes[0].set_title('Class Distribution', fontweight='bold')

# Amount by class
for cls,col,lab in [(0,'#3498db','Normal'),(1,'#e74c3c','Fraud')]:
    axes[1].hist(df[df['Class']==cls]['Amount'].clip(0,500), bins=50, alpha=0.7, color=col, label=lab)
axes[1].set_title('Transaction Amount', fontweight='bold'); axes[1].set_xlabel('Amount'); axes[1].legend()

# Correlation with Class
corr = df.corr()['Class'].abs().drop('Class').sort_values(ascending=False).head(12)
axes[2].barh(corr.index[::-1], corr.values[::-1], color='steelblue', edgecolor='black')
axes[2].set_title('Top Features (|corr| with Class)', fontweight='bold')

plt.suptitle('Credit Card Fraud — EDA', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_overview.png', bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
# Heatmap of top correlated features
top_feats = df.corr()['Class'].abs().drop('Class').sort_values(ascending=False).head(10).index.tolist()
fig, ax = plt.subplots(figsize=(10,8))
sns.heatmap(df[top_feats+['Class']].corr(), annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=ax)
ax.set_title('Correlation Matrix — Top Features + Class', fontweight='bold')
plt.tight_layout()
plt.savefig('eda_correlation.png', bbox_inches='tight', dpi=120)
plt.show()

## 🤖 Preprocessing & Model Training

In [ ]:
feature_cols = [c for c in df.columns if c != 'Class']
X = SimpleImputer(strategy='median').fit_transform(df[feature_cols])
X = StandardScaler().fit_transform(X)
y = df['Class'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

models = {
    'RF (default)':     RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1),
    'RF (balanced) ✅': RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42, n_jobs=-1)
}

trained = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    trained[name] = (model, y_pred)
    print(f'\n── {name} ──')
    print(classification_report(y_test, y_pred, target_names=['Normal','Fraud'], zero_division=0))
print('✅ Training complete')

## 📈 Evaluation

In [ ]:
best_model, best_pred = trained['RF (balanced) ✅']
y_prob = best_model.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, y_prob)
ap  = average_precision_score(y_test, y_prob)
print(f'ROC-AUC: {auc:.4f} | Avg Precision: {ap:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

cm = confusion_matrix(y_test, best_pred)
ConfusionMatrixDisplay(cm, display_labels=['Normal','Fraud']).plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix', fontweight='bold')

fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='crimson', lw=2, label=f'AUC={auc:.3f}')
axes[1].plot([0,1],[0,1],'--',color='gray'); axes[1].legend()
axes[1].set(xlabel='FPR', ylabel='TPR', title='ROC Curve')
axes[1].title.set_fontweight('bold')

prec, rec, _ = precision_recall_curve(y_test, y_prob)
axes[2].plot(rec, prec, color='steelblue', lw=2, label=f'AP={ap:.3f}')
axes[2].set(xlabel='Recall', ylabel='Precision', title='PR Curve'); axes[2].legend()
axes[2].title.set_fontweight('bold')

plt.suptitle('RF (balanced) — Fraud Detection Evaluation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('evaluation.png', bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
# Feature importance
fi = pd.DataFrame({'feature': feature_cols, 'importance': best_model.feature_importances_})
fi = fi.sort_values('importance', ascending=False).head(20)
fig, ax = plt.subplots(figsize=(9,6))
sns.barplot(data=fi, x='importance', y='feature', palette='viridis', ax=ax)
ax.set_title('Top-20 Feature Importances (RF balanced)', fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight', dpi=120)
plt.show()